# 5.14 模型不動或震盪要怎麼找步幅？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**learning rate越大，一步loss一定越小嗎？

當模型不動或loss發散，先在固定小資料試幾個步幅。小曲線找量級，再做正式比較；不要同時換架構、資料與optimizer。

**把直覺寫成數學：**一步更新可觀察L(w−ηg)；太大η可能跨過最小點。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
w = torch.tensor(1.0)
gradient = 2 * (w - 3)
for lr in (0.01, 0.1, 2.0):
    new = w - lr * gradient
    print(lr, "一步後toy loss", (new - 3).square().item())
print("模型配方：python scripts/train.py --train --lr 0.001 --steps 100")

**如何讀結果：**先用二次函數看過大步幅；模型曲線需實際跑並記錄固定seed。

**只改一件事：**只把起點w改成4。
